# 6. Evaluación, errores e interpretabilidad
**Entrada:** directorio de una ejecución real y completa. Este notebook no entrena ni fabrica resultados.
Comparar `selected` contra `baseline` por MAE en los mismos folds. Examinar RMSE, MedAE, R²,
residuos y variabilidad. La población de aplicación sigue limitada al dominio experimental.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'ingesta.py').is_file()
             and (p / 'config' / 'data_schema.yml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abra Jupyter desde el repositorio o notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import matplotlib.pyplot as plt
from src.ingesta import sha256_file

RUN_DIRECTORY = None  # results/VERSION de una ejecución real
RUN = None
if RUN_DIRECTORY:
    RUN = (ROOT / RUN_DIRECTORY).resolve()
    manifest = json.loads((RUN / 'experiment_manifest.json').read_text(encoding='utf-8'))
    if manifest.get('status') != 'complete':
        raise ValueError('La ejecución no está completa.')
    for relative, digest in manifest['artifacts_sha256'].items():
        if sha256_file(RUN / relative) != digest:
            raise ValueError(f'Artefacto modificado: {relative}')
    evaluation = json.loads((RUN / 'evaluation.json').read_text(encoding='utf-8'))
    display(pd.DataFrame(evaluation['pooled_oof_metrics']).T)
    display(pd.read_csv(RUN / 'paired_mae.csv'))
else:
    print('Sin ejecución real seleccionada; no hay métricas que mostrar.')


In [ ]:
if RUN is not None:
    oof = pd.read_csv(RUN / 'oof_predictions.csv')
    selected = oof.loc[oof['procedure'] == 'selected']
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(selected['observed'], selected['predicted'])
    bounds = [min(selected['observed'].min(), selected['predicted'].min()),
              max(selected['observed'].max(), selected['predicted'].max())]
    axes[0].plot(bounds, bounds, '--', color='gray')
    axes[0].set(xlabel='Carga observada', ylabel='Carga predicha', title='Predicción fuera de muestra')
    axes[1].scatter(selected['predicted'], selected['residual'])
    axes[1].axhline(0, color='gray', linestyle='--')
    axes[1].set(xlabel='Carga predicha', ylabel='Observada − predicha', title='Residuos externos')
    plt.tight_layout()
    plt.show()


In [ ]:
if RUN is not None:
    for fold_dir in sorted(RUN.glob('fold_*')):
        print(fold_dir.name)
        display(pd.read_csv(fold_dir / 'selected_coefficients.csv'))
        importance = fold_dir / 'selected_permutation_importance.csv'
        if importance.is_file():
            display(pd.read_csv(importance).sort_values('mae_increase_mean', ascending=False))


## Interpretar con conocimiento mecánico
Los coeficientes describen asociaciones en entradas estandarizadas. La permutación estima el
incremento del MAE al perturbar un predictor externo; la correlación y las restricciones geométricas
pueden volverla inestable o generar combinaciones poco plausibles. Comparar signos/importancias
entre folds dentro de una misma familia, sin mezclar escalas ni convertir importancia en causalidad.

SHAP, PDP/ALE e intervalos predictivos calibrados son extensiones condicionadas a modelo, datos
y supuestos; no están implementados en este avance. Revisar casos con errores grandes contra
la fuente sin modificar retrospectivamente la evaluación para mejorar el resultado.

**Entrega futura:** comparación contra baseline, incertidumbre/estabilidad con método justificado,
limitaciones de generalización y conclusiones físicas respaldadas. No extrapolar a materiales,
procesos de fabricación o condiciones de carga no evaluados.
